# Free-field comparison plots

Plot the CSV tables in `source_data/`. The observation-rate panels display 3, 5, 8, 10, 15 and 20%; the CSV also includes 2%. For downsampling, values above Q3 + 1.5 times the interquartile range are excluded before calculating the mean and population standard deviation. LNO and FNO at 4× and 5× and SDE at 6× use four trials; the other combinations use five. The CSV contains all five trial values for every combination.

Run cells in order from the repository root.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'models').is_dir() and (p / 'sampler').is_dir() and (p / 'trainer').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the MRSM repository.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
OUTPUT = ROOT / 'outputs' / 'free_field_source'
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'Arial', 'font.size': 10})

def format_current_figure():
    def tick(value, pos):
        return '0' if abs(value) < 1e-12 else f'{value:g}'
    for ax in plt.gcf().axes:
        if ax.get_xscale() == 'linear':
            ax.xaxis.set_major_formatter(FuncFormatter(tick))
        if ax.get_yscale() == 'linear':
            ax.yaxis.set_major_formatter(FuncFormatter(tick))

print('Repository:', ROOT)
print('Device:', device)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker
import matplotlib as mpl
import os
df = pd.read_csv('source_data/experiment_results_20260331_204711.csv')
plot_data = df.to_dict('list')
x_axis = plot_data['observable_pct']
_mask = np.array(x_axis) != 2
x_axis = np.array(x_axis)[_mask]
for key in plot_data:
    plot_data[key] = np.array(plot_data[key])[_mask]
if 'observable_portions_pct' in locals():
    observable_portions_pct = np.array(observable_portions_pct)[np.array(observable_portions_pct) != 2]
sde_mean = (plot_data['sde_p_rmse_mean'] + plot_data['sde_vx_rmse_mean'] + plot_data['sde_vy_rmse_mean']) / 3
sde_std = (plot_data['sde_p_rmse_std'] + plot_data['sde_vx_rmse_std'] + plot_data['sde_vy_rmse_std']) / 3
iss_mean = (plot_data['iss_p_rmse_mean'] + plot_data['iss_vx_rmse_mean'] + plot_data['iss_vy_rmse_mean']) / 3
iss_std = (plot_data['iss_p_rmse_std'] + plot_data['iss_vx_rmse_std'] + plot_data['iss_vy_rmse_std']) / 3
pi_iss_mean = (plot_data['pi_iss_p_rmse_mean'] + plot_data['pi_iss_vx_rmse_mean'] + plot_data['pi_iss_vy_rmse_mean']) / 3
pi_iss_std = (plot_data['pi_iss_p_rmse_std'] + plot_data['pi_iss_vx_rmse_std'] + plot_data['pi_iss_vy_rmse_std']) / 3
cm = 1 / 2.54
(fig, ax) = plt.subplots(figsize=(4.5 * cm, 3.8 * cm))
fs_c = 8
fs_tick = 5
fs_text = 6
mpl.rcParams.update({'font.size': fs_text, 'font.family': 'sans-serif'})
try:
    mpl.rcParams['text.usetex'] = True
    mpl.rcParams['text.latex.preamble'] = '\n        \\usepackage{amsmath}\n        \\usepackage{amssymb}\n        \\usepackage{helvet}\n        \\renewcommand{\\familydefault}{\\sfdefault}\n        \\usepackage{sansmath}\n        \\sansmath\n        \\usepackage[HTML]{xcolor}\n    '
except:
    mpl.rcParams['text.usetex'] = False
colors = {'sde': '#757575', 'iss': '#BFA05A', 'pi_iss': '#8B0000'}
ms = 3.0
lw = 0.6
cs = 2
elw = 0.4
mew = 0.4
ax.errorbar(x_axis, sde_mean, yerr=sde_std, fmt=':^', color=colors['sde'], markerfacecolor=colors['sde'], markeredgecolor='none', label='SDE', capsize=cs, markersize=ms, linewidth=lw, elinewidth=elw)
ax.errorbar(x_axis, iss_mean, yerr=iss_std, fmt='--s', color=colors['iss'], markerfacecolor=colors['iss'], markeredgecolor='none', label='ISS', capsize=cs, markersize=ms, linewidth=lw, elinewidth=elw)
ax.errorbar(x_axis, pi_iss_mean, yerr=pi_iss_std, fmt='-o', color=colors['pi_iss'], markerfacecolor='white', markeredgecolor=colors['pi_iss'], markeredgewidth=mew * 1.5, label='\\textbf{\\color[HTML]{8B0000}{PI-ISS}}', capsize=cs, markersize=ms * 1.2, linewidth=lw * 2.0, elinewidth=elw * 1.5)
ax.set_xlabel('Observable portion (\\%)', fontsize=fs_text)
ax.set_ylabel('nRMSE', fontsize=fs_text)
ax.set_title('Errors (nRMSE)', fontsize=fs_text)
y_ticks = [0, 0.05, 0.1, 0.15]
ax.set_yticks(y_ticks)
ax.set_yticklabels(['0', '0.05', '0.1', '0.15'])
if 'observable_portions_pct' in locals():
    ax.set_xticks(observable_portions_pct)
else:
    ax.set_xticks(x_axis)
ax.set_ylim(0, max(np.max(sde_mean + sde_std), 0.1) * 1.1)
ax.tick_params(axis='both', which='major', labelsize=fs_tick, length=2, pad=1)
ax.set_xlim(left=min(x_axis) - 1, right=max(x_axis) + 1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)
ax.grid(False)
ax.legend(ncol=1, loc='upper right', bbox_to_anchor=(0.8, 0.95), frameon=False, fontsize=fs_text, labelspacing=0.6, handletextpad=0.3, borderaxespad=0)
plt.tight_layout(pad=0.2)
save_dir = 'outputs/free_field_source/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, 'Average_nRMSE_Acoustic.eps')
plt.savefig(save_path, dpi=300, format='eps', bbox_inches='tight')
format_current_figure()
plt.show()
print(f'\nSaved figure: {save_path}')


In [ ]:
import matplotlib.pyplot as plt
import os
import numpy as np
import matplotlib.ticker as ticker
import pandas as pd
import matplotlib as mpl
csv_path = 'source_data/experiment_results_20260331_204711.csv'
df = pd.read_csv(csv_path)
plot_data = df.to_dict('list')
x_axis = np.array(plot_data['observable_pct'])
_mask = x_axis != 2
x_axis = x_axis[_mask]
for key in plot_data:
    plot_data[key] = np.array(plot_data[key])[_mask]
observable_portions_pct = x_axis
sde_corr_mean = (plot_data['sde_p_corr_mean'] + plot_data['sde_vx_corr_mean'] + plot_data['sde_vy_corr_mean']) / 3
sde_corr_std = (plot_data['sde_p_corr_std'] + plot_data['sde_vx_corr_std'] + plot_data['sde_vy_corr_std']) / 3
iss_corr_mean = (plot_data['iss_p_corr_mean'] + plot_data['iss_vx_corr_mean'] + plot_data['iss_vy_corr_mean']) / 3
iss_corr_std = (plot_data['iss_p_corr_std'] + plot_data['iss_vx_corr_std'] + plot_data['iss_vy_corr_std']) / 3
pi_iss_corr_mean = (plot_data['pi_iss_p_corr_mean'] + plot_data['pi_iss_vx_corr_mean'] + plot_data['pi_iss_vy_corr_mean']) / 3
pi_iss_corr_std = (plot_data['pi_iss_p_corr_std'] + plot_data['pi_iss_vx_corr_std'] + plot_data['pi_iss_vy_corr_std']) / 3
cm = 1 / 2.54
(fig, ax) = plt.subplots(figsize=(4.5 * cm, 3.8 * cm))
fs_tick = 5
fs_text = 6
mpl.rcParams.update({'font.size': fs_text, 'font.family': 'sans-serif'})
try:
    mpl.rcParams['text.usetex'] = True
    mpl.rcParams['text.latex.preamble'] = '\n        \\usepackage{amsmath}\n        \\usepackage{amssymb}\n        \\usepackage{helvet}\n        \\renewcommand{\\familydefault}{\\sfdefault}\n        \\usepackage{sansmath}\n        \\sansmath\n    '
except:
    mpl.rcParams['text.usetex'] = False
colors = {'sde': '#757575', 'iss': '#BFA05A', 'pi_iss': '#8B0000'}
ms = 3.0
lw = 0.6
cs = 2
elw = 0.4
mew = 0.4
ax.errorbar(x_axis, sde_corr_mean, yerr=sde_corr_std, fmt=':^', color=colors['sde'], markerfacecolor=colors['sde'], markeredgecolor='none', label='SDE', capsize=cs, markersize=ms, linewidth=lw, elinewidth=elw)
ax.errorbar(x_axis, iss_corr_mean, yerr=iss_corr_std, fmt='--s', color=colors['iss'], markerfacecolor=colors['iss'], markeredgecolor='none', label='ISS', capsize=cs, markersize=ms, linewidth=lw, elinewidth=elw)
ax.errorbar(x_axis, pi_iss_corr_mean, yerr=pi_iss_corr_std, fmt='-o', color=colors['pi_iss'], markerfacecolor='white', markeredgecolor=colors['pi_iss'], markeredgewidth=mew * 1.5, label='\\textbf{PI-ISS}', capsize=cs, markersize=ms * 1.2, linewidth=lw * 2.0, elinewidth=elw * 1.5)
ax.set_xlabel('Observable portion (\\%)', fontsize=fs_text)
ax.set_ylabel('Correlation', fontsize=fs_text)
ax.set_title('Pearson correlation coefficient', fontsize=fs_text)
y_ticks = [0.98, 0.99, 1.0]
ax.set_yticks(y_ticks)
ax.set_yticklabels(['0.98', '0.99', '1'])
ax.set_xticks(observable_portions_pct)
ax.tick_params(axis='both', which='major', labelsize=fs_tick, length=2, pad=1)
ax.set_xlim(left=min(x_axis) - 1, right=max(x_axis) + 1)
ax.set_ylim(0.988, 1.0005)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)
ax.grid(False)
leg = ax.legend(ncol=1, loc='lower right', bbox_to_anchor=(0.8, 0.2), frameon=False, fontsize=fs_text, labelspacing=0.6, handletextpad=0.3, borderaxespad=0)
for text in leg.get_texts():
    if 'PI-ISS' in text.get_text():
        text.set_color(colors['pi_iss'])
plt.tight_layout(pad=0.2)
save_dir = 'outputs/free_field_source/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, 'Average_PCC_Acoustic.eps')
plt.savefig(save_path, dpi=300, format='eps', bbox_inches='tight')
format_current_figure()
plt.show()
print(f'\nSaved correlation figure: {save_path}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
from matplotlib.ticker import FuncFormatter
import os
csv_file_path = 'source_data/baselines_comparison_20260331_213441.csv'
df = pd.read_csv(csv_file_path)
SCALES = df['Scale'].tolist()
methods_ordered = ['LNO', 'FNO', 'SDE', 'ISS', 'PI-ISS']
means = {m: [] for m in methods_ordered}
stds = {m: [] for m in methods_ordered}
for m in methods_ordered:
    trial_cols = [f'{m}_trial_{i}' for i in range(1, 6)]
    for (idx, row) in df.iterrows():
        raw_vals = row[trial_cols].values.astype(float)
        valid_vals = raw_vals[~np.isnan(raw_vals)]
        if len(valid_vals) >= 3:
            q1 = np.percentile(valid_vals, 25)
            q3 = np.percentile(valid_vals, 75)
            iqr = q3 - q1
            upper_bound = q3 + 1.5 * iqr
            clean_vals = valid_vals[valid_vals <= upper_bound]
        else:
            clean_vals = valid_vals
        if len(clean_vals) > 0:
            means[m].append(np.mean(clean_vals))
            stds[m].append(np.std(clean_vals))
        else:
            means[m].append(0.0)
            stds[m].append(0.0)
FIG_WIDTH = 12.5 / 2.54
FIG_HEIGHT = 3.5 / 2.54
FONT_SIZE_TEXT = 6
FONT_SIZE_LABEL = 8
mpl.rcParams.update({'font.size': FONT_SIZE_TEXT, 'axes.linewidth': 0.25, 'xtick.major.width': 0.2, 'ytick.major.width': 0.2, 'font.family': 'sans-serif'})
try:
    mpl.rcParams['text.usetex'] = True
    mpl.rcParams['text.latex.preamble'] = '\n        \\usepackage{amsmath}\n        \\usepackage{amssymb}\n        \\usepackage{helvet}\n        \\renewcommand{\\familydefault}{\\sfdefault}\n        \\usepackage{sansmath}\n        \\sansmath\n    '
except:
    mpl.rcParams['text.usetex'] = False
color_list = ['#E0E0E0', '#BDBDBD', '#757575', '#BFA05A', '#8B0000']
(fig, ax) = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT), layout='constrained')
x = np.arange(len(SCALES))
width = 0.15
multiplier = 0
for (i, method) in enumerate(methods_ordered):
    offset = width * multiplier
    current_color = color_list[i]
    label_text = '\\textbf{PI-ISS}' if method == 'PI-ISS' else method
    ax.bar(x + offset, means[method], width, yerr=stds[method], label=label_text, color=current_color, capsize=1.5, edgecolor='#333333', linewidth=0.2, error_kw={'elinewidth': 0.4, 'capthick': 0.4, 'ecolor': '#333333'})
    multiplier += 1
for spine in ax.spines.values():
    spine.set_linewidth(0.25)
ax.set_ylabel('nRMSE', fontsize=fs_text)
ax.set_xlabel('Downsampling factor', fontsize=fs_text)
ax.set_title('Error evaluation at varying sparsity', fontsize=fs_text, y=0.98, x=0.45)
ax.set_xticks(x + width * 2)
ax.set_xticklabels([f'{s}×' for s in SCALES], fontsize=fs_text)
ax.tick_params(axis='y', labelsize=fs_tick, width=0.2, length=1.5, pad=1)
ax.tick_params(axis='x', labelsize=fs_tick, width=0.2, length=0, pad=1)
leg = ax.legend(loc='upper left', fontsize=fs_text, frameon=False, ncols=5, handletextpad=0.2, columnspacing=0.8)
for text in leg.get_texts():
    if 'PI-ISS' in text.get_text():
        text.set_color('#8B0000')
ax.set_ylim(0, 0.4)

def custom_zero_formatter(y, pos):
    if abs(y) < 1e-09:
        return '0'
    return f'{y:g}'
ax.yaxis.set_major_formatter(FuncFormatter(custom_zero_formatter))
save_dir = 'outputs/free_field_source/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, 'error_comparison_filtered.eps')
plt.savefig(save_path, format='eps', dpi=300)
format_current_figure()
plt.show()
print(f'Saved downsampling figure: {save_path}.')
